# Объяснение кода

### async_client.py

In [ ]:
async def send_messages(writer):
    """
    Считывает ввод пользователя и отправляет на сервер.

    :param writer: Объект потока записи.
    """
    while True:
        message = await asyncio.to_thread(sys.stdin.readline)
        if not message:
            break
        writer.write(message.encode())
        await writer.drain()

Считывает ввод пользователя с консоли и отправляет на сервер

async def - определение асинхронной функции (корутины). Внутри можно использовать await
writer - объект потока записи (asyncio.StreamWriter), который отправляет данные на сервер

Бесконечный цикл - функция будет работать, пока пользователь не завершит программу

sys.stdin.readline - синхронная функция чтения строки из стандартного ввода (консоли). Она блокирует выполнение, пока пользователь не нажмет Enter
asyncio.to_thread() - ключевое средство:
- Запускает синхронную функцию в отдельном потоке (из пула потоков)
- Не блокирует основной асинхронный цикл (другие задачи продолжают работать)
- Возвращает awaitable объект, поэтому используется await
Зачем? - sys.stdin.readline() блокирующая операция. В асинхронном коде мы не можем просто вызвать её - это остановило бы все другие корутины. to_thread оборачивает это в отдельный поток.

Если readline вернула пустую строку (EOF - конец файла, например при Ctrl+D или закрытии терминала) - выходим из цикла

message.encode() - преобразует строку в байты (по умолчанию UTF-8). Сеть работает с байтами, а не строками
writer.write() - неблокирующая запись байтов в буфер отправки. Ставит данные в очередь на отправку

drain() - ожидает, пока буфер отправки освободится и данные реально уйдут в сеть.
Это важно для управления потоком (backpressure) - чтобы не отправлять данные быстрее, чем уходит по сети

In [ ]:
async def receive_messages(reader):
    """
    Принимает сообщения от сервера и выводит в консоль.

    :param reader: Объект потока чтения.
    """
    while True:
        data = await reader.read(1024)
        if not data:
            print("\nСоединение с сервером разорвано.")
            break
        print(f"\nСервер: {data.decode().strip()}")
        print("Вы: ", end="", flush=True)

Принимает сообщения от сервера и выводит их в консоль

Асинхронная функция, принимает reader - asyncio.StreamReader для чтения из сокета

Бесконечный цикл приема сообщений

reader.read(1024) - асинхронное чтение до 1024 байт из сокета
await - ждем, пока данные придут по сети или соединение не закроется
Возвращает bytes объект. Если соединение закрыто - возвращает пустые байты

Если данные не получены (пустой bytes) - сервер закрыл соединение. Выводим сообщение и выходим из цикла

data.decode() - преобразует байты в строку (UTF-8)
.strip() - удаляет символы перевода строки в начале/конце
Выводим полученное сообщение

end="" - не переходить на новую строку после вывода
flush=True - принудительно выводим на экран сразу (без буферизации)
Это делается, чтобы после получения сообщения от сервера курсор оставался на строке ввода пользователя

In [ ]:
async def main():
    """Подключение к серверу и запуск цикла обмена сообщениями."""
    try:
        reader, writer = await asyncio.open_connection('127.0.0.1', 8888)
        print("Подключено к серверу. Введите сообщение:")

        await asyncio.gather(
            receive_messages(reader),
            send_messages(writer)
        )
    except ConnectionRefusedError:
        print("Ошибка: Не удалось подключиться к серверу. Проверьте, запущен ли он.")
    except Exception as e:
        print(f"Произошла ошибка: {e}")
    finally:
        if 'writer' in locals():
            writer.close()
            await writer.wait_closed()

Устанавливает подключение и запускает параллельную работу приема/отправки

Главная асинхронная корутина

asyncio.open_connection() - асинхронно устанавливает TCP-соединение
Параметры: host='127.0.0.1' (localhost), port=8888
Возвращает кортеж (reader, writer):
- reader - для чтения данных из сокета
- writer - для записи данных в сокет
await - ждём, пока соединение реально установится

Информационное сообщение

asyncio.gather() - важное средство для параллельного выполнения корутин
Принимает несколько awaitable объектов и запускает их конкурентно
Как это работает:
- receive_messages и send_messages выполняются практически одновременно
- Если одна корутина блокируется (await reader.read() ждёт данные), вторая продолжает работу
- gather завершится, когда все переданные корутины завершатся
Почему не asyncio.create_task? - gather удобнее, когда нужно дождаться завершения всех задач

Специфическая ошибка, если сервер не запущен или порт закрыт

Ловим любые другие исключения (сетевая ошибка, таймаут и т.д.)

locals() - словарь локальных переменных. Проверяем, была ли создана переменная writer (подключение могло не установиться)
writer.close() - инициирует закрытие соединения
await writer.wait_closed() - ожидаем реального закрытия (важно для корректного освобождения ресурсов)

In [ ]:
if __name__ == "__main__":
    try:
        asyncio.run(main())
    except KeyboardInterrupt:
        print("\nВыход из чата.")

asyncio.run(main()) - точка входа в асинхронную программу:
- Создаёт новый event loop
- Запускает корутину main() до её завершения
- После завершения закрывает event loop
- Важно: может вызываться только один раз в программе на одном потоке

KeyboardInterrupt - возникает при Ctrl+C. Выводим сообщение, программа завершается

### async_server.py

In [ ]:
async def send_messages(writer):
    """
    Считывает ввод из консоли и отправляет его клиенту.

    :param writer: Объект потока записи asyncio.
    """
    while True:
        # Используем to_thread, чтобы input() не блокировал event loop
        message = await asyncio.to_thread(sys.stdin.readline)
        if not message:
            break
        writer.write(message.encode())
        await writer.drain()

Читает ввод сервера и отправляет сообщение клиенту

async def - асинхронная функция (корутина). writer - asyncio.StreamWriter для отправки данных конкретному клиенту

Бесконечный цикл - сервер может отправлять множество сообщений одному клиенту

sys.stdin.readline() - блокирующий вызов, ждет пока сервер введет сообщение и нажмет Enter. asyncio.to_thread() - запускает синхронную функцию в отдельном потоке из пула. Если бы мы вызвали sys.stdin.readline() напрямую, весь event loop замер бы в ожидании ввода. Сервер не смог бы обслуживать других клиентов или принимать новые подключения. await - ждем завершения потока и получения строки. 

Если получили пустую строку (EOF, например Ctrl+D) - выходим из цикла

message.encode() - строка → байты (UTF-8). writer.write() - буферизированная запись в сокет клиента

Ждем, пока данные реально отправятся клиенту. Предотвращает переполнение буфера отправки. Если клиент медленно читает (или отключился), drain() приостановит отправку. 

In [ ]:
async def receive_messages(reader):
    """
    Принимает сообщения от клиента и выводит их в консоль.

    :param reader: Объект потока чтения asyncio.
    """
    while True:
        data = await reader.read(1024)
        if not data:
            print("\nКлиент отключился.")
            break
        print(f"\nКлиент: {data.decode().strip()}")
        print("Вы: ", end="", flush=True)

Читает сообщения от клиента и выводит их на экран сервера.

reader - asyncio.StreamReader для чтения данных от клиента

Бесконечный цикл приема сообщений от клиента

Асинхронно читаем до 1024 байт из сокета. await - если данных нет, корутина засыпает, event loop переключается на другие задачи. Возвращает:
- Байты с данными, если что-то пришло
- Пустые байты b'', если клиент закрыл соединение

Пустые байты = соединение разорвано. Выводим сообщение и выходим из цикла. 

data.decode() - байты → строка (UTF-8). .strip() - удаляем лишние пробелы и переводы строк. Выводим сообщение от клиента (с новой строки). 

Возвращаем приглашение ко вводу. end="" - не переводим строку. flush=True - немедленный вывод (без буферизации). Создает эффект, что сервер готов писать следующее сообщение

In [ ]:
async def handle_client(reader, writer):
    """
    Управляет соединением с клиентом.

    :param reader: Объект потока чтения.
    :param writer: Объект потока записи.
    """
    addr = writer.get_extra_info('peername')
    print(f"Новое соединение от {addr}")
    print("Вы можете писать сообщения...")

    # Запускаем чтение и отправку параллельно
    try:
        await asyncio.gather(
            receive_messages(reader),
            send_messages(writer)
        )
    except Exception as e:
        print(f"Ошибка: {e}")
    finally:
        writer.close()
        await writer.wait_closed()

Управляет соединением с одним клиентом - координатор между чтением и отправкой.

Это корутина-обработчик для каждого подключения. Вызывается автоматически сервером при новом подключении. 

get_extra_info('peername') - получает информацию о подключении. peername - кортеж (ip_адрес, порт) удаленного клиента. Информируем сервер о новом клиенте

asyncio.gather() - ключевой момент сервера. Запускает две корутины конкурентно:
- receive_messages - читает от клиента и выводит на экран
- send_messages - читает ввод оператора и отправляет клиенту

await gather - ждет, пока обе корутины не завершатся. Если одна завершится (например, клиент отключился - receive_messages выйдет), gather не прервет вторую сразу, но получит исключение или дождется ее

Ловим любые ошибки при обмене (разрыв соединения, ошибки декодирования и т.д.)

writer.close() - инициирует закрытие TCP-соединения. await writer.wait_closed() - ждем реального закрытия (важно для корректного освобождения ресурсов ОС). 

In [ ]:
async def main():
    """Запуск сервера."""
    server = await asyncio.start_server(handle_client, '127.0.0.1', 8888)
    addr = server.sockets[0].getsockname()
    print(f"Сервер запущен на {addr}")

    async with server:
        await server.serve_forever()

Запускает TCP-сервер и начинает обслуживание

Главная асинхронная корутина

asyncio.start_server() - создает TCP-сервер. Параметры:
- handle_client - функция-обработчик для каждого клиента
- '127.0.0.1' - слушаем только локальные подключения
- 8888 - порт

Возвращает объект asyncio.Server. await - ждем, пока сервер реально начнет слушать порт.

server.sockets - список сокетов, на которых слушает сервер. [0] - берем первый (обычно единственный) сокет. getsockname() - получаем реальный адрес и порт, на которых сервер запустился. Возвращает кортеж ('127.0.0.1', 8888). 

Информируем администратора

async with server - контекстный менеджер для сервера.
- При входе - сервер готов к работе
- При выходе - сервер корректно закрывается

server.serve_forever() - запускает основной цикл обработки подключений
Блокирует выполнение (но асинхронно, не блокируя event loop)
Работает, пока не будет остановлен (например, Ctrl+C)
Принимает новые подключения и для каждого создает задачу с handle_client

In [ ]:
if __name__ == "__main__":
    try:
        asyncio.run(main())
    except KeyboardInterrupt:
        print("\nСервер остановлен.")

asyncio.run(main()) - точка входа в асинхронную программу
- Создает event loop
- Запускает main() до завершения
- Корректно закрывает loop после завершения

Обработка Ctrl+C - плавная остановка сервера

Reader - объект для асинхронного чтения данных из сокета. Writer - для асинхронной записи. Работают как "водопроводные краны": reader принимает входящий поток, writer отправляет исходящий.

Что такое flush и зачем он нужен? Flush - принудительная отправка накопленных в буфере данных на экран/в файл. Нужен, чтобы данные появились немедленно, без задержки из-за буферизации.

Как get_extra_info('peername') делает кортеж? Никак. 'peername' - это ключ (строка). Метод по этому ключу возвращает уже готовый кортеж (ip, port) от системного вызова getpeername().

Что такое gather? Запускает несколько асинхронных задач параллельно (конкурентно) и ждет завершения всех. Возвращает список результатов в том же порядке.

Что делает wait_closed? Асинхронно ждет, пока соединение полностью не закроется на уровне ОС: все данные отправлены, буфер очищен, сокет освобожден.

"Асинхронно заблокировать выполнение, не блокируя event loop"? Останавливает только текущую корутину (она "спит"), но event loop продолжает выполнять другие задачи. Поток ОС не блокируется.

Что значит "асинхронно сделать что-то"? Коротко: Асинхронно = "начал операцию, но не жду её завершения прямо сейчас. Могу делать другие дела, а когда операция закончится и что-то будет получено - мне сообщат". Аналогия:
- Синхронно: Позвонил в техподдержку и висишь на линии 30 минут, ничего не делая
- Асинхронно: Оставил заявку, тебе перезвонят. Пока ждешь - пьешь чай, работаешь, гуляешь

### collect_info.py

In [ ]:
class Composer:
    """Класс для агрегации данных, полученных от сервиса."""

    def __init__(self):
        """Инициализация аккумулятора результата."""
        self.result = 0
        self._lock = asyncio.Lock()

    async def add(self, data: dict):
        """
        Добавляет значение из словаря к общему результату.

        :param data: Словарь, содержащий ключ 'data'.
        """
        async with self._lock:
            self.result += data.get('data', 0)

Накопление суммы чисел из ответов сервера

Простое объявление класса с docstring.

self.result = 0 - создает атрибут экземпляра для хранения накопленной суммы. Никакой асинхронности здесь нет.

data: dict - аннотация типа: ожидаем словарь. data.get('data', 0) - безопасное извлечение значения по ключу 'data'. Если ключа нет - возвращаем 0. self.result += ... - атомарное сложение (в асинхронном контексте это важно, но здесь все задачи добавляют через одну и ту же ссылку на композитор). 

In [ ]:
async def fetch(
    session: aiohttp.ClientSession,
    url: str,
    param: int,
    composer: Composer,
    semaphore: asyncio.Semaphore
):
    """
    Выполняет асинхронный GET-запрос и передает данные в Composer.

    :param session: Сессия aiohttp для выполнения запросов.
    :param url: Базовый URL сервиса.
    :param param: Параметр запроса.
    :param composer: Экземпляр класса Composer для сбора данных.
    :param semaphore: Семафор для ограничения количества одновременных запросов.
    """
    async with semaphore:
        try:
            async with session.get(f'{url}/{param}') as response:
                if response.status == 200:
                    data = await response.json()
                    await composer.add(data)
                else:
                    # Можно добавить логирование ошибок сервера
                    pass
        except Exception as e:
            # Обработка сетевых ошибок
            print(f"Ошибка при запросе {param}: {e}")

Выполнить один HTTP GET-запрос и добавить результат в композитор

async def - асинхронная корутина. session: aiohttp.ClientSession - переиспользуемая сессия для HTTP-запросов (TCP Keep-Alive). param: int - параметр, подставляемый в URL (например, http://localhost:8003/42). composer: Composer - экземпляр класса для накопления результата. semaphore: asyncio.Semaphore - семафор для ограничения количества одновременных запросов. 

async with semaphore - вход в контекстный менеджер семафора. Семафор имеет внутренний счетчик (здесь 500). Когда 500 корутин уже внутри блока, 501-я приостанавливается (await) на этом месте. Как только одна корутина выходит из блока, семафор пропускает следующую. Без семафора: 100 000 одновременных запросов →
- Исчерпание файловых дескрипторов
- Перегрузка сетевой подсистемы
- Скорее всего, сервер не выдержит

session.get(f'{url}/{param}') - создает GET-запрос, но не отправляет его. async with ... as response - асинхронный контекстный менеджер:
- При входе: отправляет запрос и ждет заголовки ответа (await внутри)
- При выходе: закрывает соединение (возвращает в пул)

response.status == 200 - проверка HTTP статуса (OK). await response.json() - асинхронно читаем тело ответа и парсим JSON
- Метод читает поток данных до конца (может быть несколько read())
- Возвращает Python-объект (обычно dict/list)

await composer.add(data) - асинхронно добавляем значение к общей сумме

Место для обработки ошибок HTTP (404, 500 и т.д.)

Ловим любые исключения (сетевые ошибки, таймауты, разрыв соединения). Не прерываем выполнение других запросов. Просто выводим ошибку и продолжаем работу. 

In [ ]:
async def collector(params: range) -> int:
    """
    Основная асинхронная функция для сбора данных по списку параметров.

    :param params: Список или диапазон параметров для запросов.
    :return: Итоговая сумма собранных данных.
    """
    url = 'http://localhost:8003'
    composer = Composer()
    # Ограничиваем количество одновременных соединений (например, 500),
    # чтобы не исчерпать ресурсы системы и не уронить сервер.
    semaphore = asyncio.Semaphore(500)

    # Используем ClientSession для повторного использования соединений (TCP Keep-Alive)
    async with aiohttp.ClientSession() as session:
        tasks = [
            fetch(session, url, i, composer, semaphore)
            for i in params
        ]
        # Запускаем все задачи конкурентно
        await asyncio.gather(*tasks)

    return composer.result

Оркестрация всех запросов и сбор итогового результата

params: range - принимает диапазон (например, range(100000)). -> int - возвращает итоговую сумму. 

Базовый URL сервера. Создаем экземпляр композитора для накопления суммы. 

Создаем семафор с пределом 500 одновременных запросов. Почему 500?
- Слишком мало (10) - запросы выполняются медленно (не используем параллелизм)
- Слишком много (10000) - убиваем систему (открытые сокеты, память, CPU)
- 500 - разумный компромисс для большинства систем

ClientSession() - создает сессию для HTTP-запросов. Преимущества сессии: 
- CP Keep-Alive - переиспользование соединений (не создаем новое на каждый запрос)
- Пул соединений - ограничивает количество одновременных подключений
- Общие заголовки - можно задать раз для всех запросов

async with - гарантирует закрытие сессии после всех запросов

Генератор списка задач. На этом этапе корутины не запускаются! fetch(...) просто создает объекты корутин (специальные генераторы). Они начнут выполняться только внутри gather().

*tasks - распаковка списка в аргументы (как gather(tasks[0], tasks[1], ...)). asyncio.gather() - запускает все 100 000 корутин конкурентно. await - ждет завершения всех задач (успешных или с ошибками). Важный нюанс: Несмотря на 100 000 задач, одновременно выполняется только 500 (благодаря семафору внутри fetch). Остальные ждут своей очереди внутри async with semaphore.

Возвращаем накопленную сумму

In [ ]:
def main():
    # Входные параметры
    request_params = range(100_000)

    # Запуск асинхронного цикла
    final_result = asyncio.run(collector(request_params))
    print(f"Итоговый результат: {final_result}")

range(100_000) - создает диапазон от 0 до 99 999. Используем _ для улучшения читаемости больших чисел. 

asyncio.run() - создает event loop, запускает collector(), ждет результата, закрывает loop. Возвращает результат корутины (в данном случае int). 

Выводим сумму